# 01 — EDA (Home Credit Default Risk)

**Objetivo de este notebook:** caracterizar la tabla principal `application_*`, el `TARGET`, la calidad de datos (nulos, duplicados, rangos plausibles), la integridad de IDs frente al test y documentar **riesgos de leakage** a vigilar en features agregadas. Los modelos y el split formal viven en notebooks posteriores; aquí solo fijamos el contexto y el diagnóstico exploratorio.

**Problema (Kaggle — Home Credit Default Risk):** predecir si un solicitante **incumplirá** el préstamo (`TARGET` = 1). La métrica oficial de envío es **ROC-AUC** entre la probabilidad predicha y el `TARGET` observado (solo en train).

## Fuentes de datos

| Recurso | Uso |
|--------|-----|
| `data/raw/*.csv` | CSV del competición (no versionar pesados; ver `.gitignore`) |
| `configs/home_credit.yaml` | Nombres de archivo, claves (`SK_ID_CURR`, …), columna `TARGET` |
| `configs/base.yaml` | Semilla, proporciones de split (referencia para reproducibilidad) |
| `HomeCredit_columns_description.csv` | Diccionario de columnas por tabla |

En código: `src.data.home_credit` (`read_table`, `table_path`, `target_column`, …).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Raíz del repo (notebook puede ejecutarse desde `notebooks/` o la raíz)
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display

from src.data.home_credit import keys, read_table, table_path, target_column
from src.data.quality import basic_range_checks, missing_summary
from src.utils.paths import project_root
from src.utils.seed import set_seed

plt.style.use("ggplot")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

set_seed(42)
print("project_root:", project_root())

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
train = read_table("application_train")
test = read_table("application_test")

y_col = target_column()
k_curr = keys()["sk_id_curr"]

print("train:", train.shape, "| test:", test.shape)
print("TARGET en train:", y_col in train.columns)
print("TARGET en test:", y_col in test.columns)

## Tabla principal `application_train` / `application_test`

- Una fila por solicitud (`SK_ID_CURR`).
- `TARGET` **solo en train** (0 = no default, 1 = default en la ventana observada).
- Mezcla de numéricas, categóricas codificadas y flags binarios.

In [ ]:
vc = train[y_col].value_counts().sort_index()
imbalance = (vc.get(1, 0) / len(train)) if len(train) else 0
print("Distribución TARGET:\n", vc)
print(f"Proporción positivos (TARGET=1): {imbalance:.4f}")

fig, ax = plt.subplots(figsize=(5, 3))
vc.plot(kind="bar", ax=ax, color=["#4c78a8", "#f58518"])
ax.set_xlabel("TARGET")
ax.set_ylabel("Recuento")
ax.set_title("Distribución de clases (train)")
plt.tight_layout()

### Tipos y columnas

Resumen rápido de dtypes; las categóricas de alta cardinalidad suelen requerir encoding cuidadoso más adelante.

In [ ]:
dtypes_train = train.dtypes.value_counts()
print(dtypes_train)

# Excluir TARGET del análisis de features
feat_cols = [c for c in train.columns if c != y_col]
obj_cols = train[feat_cols].select_dtypes(include=["object"]).columns.tolist()
print(f"Columnas tipo object (candidatas a categórica): {len(obj_cols)}")

card = train[obj_cols].nunique(dropna=False).sort_values(ascending=False)
print("Top 15 por cardinalidad (object):")
print(card.head(15))

## Valores faltantes

Priorizamos columnas con mayor fracción de `NA` en train (mismas columnas suelen existir en test con patrones parecidos).

In [ ]:
miss = missing_summary(train).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f"Columnas con al menos un nulo: {len(miss_nonzero)} / {len(train.columns)}")
display(miss_nonzero.head(25).to_frame("frac_na"))

fig, ax = plt.subplots(figsize=(8, 5))
miss_nonzero.head(25).plot(kind="barh", ax=ax, legend=False, color="#4c78a8")
ax.set_xlabel("Fracción NA")
ax.set_title("Top 25 columnas por % de nulos (train)")
plt.tight_layout()

## Calidad: duplicados e IDs

- `SK_ID_CURR` debe ser **único** por fila en train y en test.
- Train y test no deben **compartir** el mismo `SK_ID_CURR` (conjuntos disjuntos para simular el holdout de Kaggle).

In [ ]:
dup_train = train[k_curr].duplicated().sum()
dup_test = test[k_curr].duplicated().sum()
overlap = set(train[k_curr]) & set(test[k_curr])

print(f"Filas duplicadas por {k_curr} en train: {dup_train}")
print(f"Filas duplicadas por {k_curr} en test: {dup_test}")
print(f"IDs compartidos train∩test: {len(overlap)}")

## Otras tablas (volumen relativo)

Las tablas de comportamiento tienen **múltiples filas por cliente**; aquí solo contamos filas para dimensionar el trabajo de agregación en el notebook 02.

In [ ]:
def count_csv_rows(path: Path) -> int:
    """Número de filas de datos (excl. cabecera), sin cargar todo en memoria."""
    with path.open("rb") as f:
        return sum(1 for _ in f) - 1


aux_keys = [
    "bureau",
    "bureau_balance",
    "previous_application",
    "pos_cash_balance",
    "installments_payments",
    "credit_card_balance",
]
rows = {}
for k in aux_keys:
    p = table_path(k)
    rows[k] = count_csv_rows(p)

pd.Series(rows, name="n_rows").sort_values(ascending=False)

## Rangos plausibles (sanity checks)

Comprobaciones ligeras sobre columnas con significado conocido; un `False` merece revisión en el diccionario de columnas.

In [ ]:
# Rangos orientativos (dataset Kaggle; ajustar si el CSV local difiere)
range_hints = {
    "EXT_SOURCE_1": (0.0, 1.0),
    "EXT_SOURCE_2": (0.0, 1.0),
    "EXT_SOURCE_3": (0.0, 1.0),
    "DAYS_BIRTH": (-25000, -1),
    "DAYS_EMPLOYED": (-25000, 365243),  # valor especial 365243 = "XNA" en muchas ediciones
}

present = {k: v for k, v in range_hints.items() if k in train.columns}
chk = basic_range_checks(train, present)
for col, ok in sorted(chk.items()):
    print(f"{col}: {'OK' if ok else 'REVISAR'}")

if "DAYS_EMPLOYED" in train.columns:
    special = (train["DAYS_EMPLOYED"] == 365243).mean()
    print(f"Fracción DAYS_EMPLOYED == 365243 (placeholder): {special:.4f}")

## Asociación univariada con `TARGET` (numéricas)

Solo orientativo: correlación de Pearson con el `TARGET`; no implica causalidad ni ausencia de interacciones.

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if y_col in num_cols:
    num_cols.remove(y_col)

corr = train[num_cols + [y_col]].corr(numeric_only=True)[y_col].drop(labels=[y_col], errors="ignore")
corr = corr.replace([np.inf, -np.inf], np.nan).dropna()
corr_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)

print("Top 15 |corr| con TARGET:")
display(corr_sorted.head(15).to_frame("corr_with_TARGET"))
print("\nBottom 5 (menor asociación lineal):")
display(corr_sorted.tail(5).to_frame("corr_with_TARGET"))

## Auditoría preliminar de leakage (nivel `application_*`)

**Qué revisamos aquí:** ausencia de etiquetas o estadísticas calculadas *después* del evento en columnas de la tabla principal; nombres que sugieran fuga de información.

**Qué no resolvemos aún:** al agregar `bureau`, `previous_application`, balances y cuotas, hay que definir **punto de corte temporal** (solo información disponible en la fecha de solicitud) y evitar estadísticas que mezclen estados posteriores al outcome. Eso se documenta en el informe técnico (sección *Leakage audit*) y se valida al construir features en el notebook 02.

In [ ]:
# Columnas cuyo nombre sugiere revisión manual (heurística)
sus = [
    c
    for c in train.columns
    if any(
        x in c.upper()
        for x in (
            "TARGET",
            "DEFAULT",
            "FUTURE",
            "POST",
        )
    )
]
# TARGET es la etiqueta; el resto merece lectura en HomeCredit_columns_description.csv
print("Columnas a revisar por nombre:", [c for c in sus if c != y_col])

doc_flags = [c for c in train.columns if "FLAG_DOCUMENT" in c]
print(f"\nFLAG_DOCUMENT_* (documentación aportada, no es 'leakage' por sí sola): {len(doc_flags)} columnas")

## Reproducibilidad (referencia al pipeline)

Los experimentos del repo usan la semilla y los splits definidos en `configs/base.yaml` (validación cruzada y holdout en notebooks posteriores). Aquí solo mostramos los valores para trazabilidad del EDA.

In [ ]:
base_path = project_root() / "configs" / "base.yaml"
with open(base_path, encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("seed:", base_cfg.get("seed"))
print("splits:", base_cfg.get("splits"))
print("métricas configuradas:", base_cfg.get("metrics"))

## Resumen y siguientes pasos

| Hallazgo típico | Implicación |
|-----------------|-------------|
| Clase positiva minoritaria | Estratificar splits / CV; mirar PR-AUC y calibración además de ROC-AUC |
| Muchos nulos por bloques (p. ej. bienes) | Imputación o modelos que toleran NaN; coherencia train/test |
| Tablas auxiliares muy largas | Agregaciones por `SK_ID_CURR` con criterio temporal claro |
| `EXT_SOURCE_*` y `DAYS_*` con correlación notable | Buen candidato a features; validar rangos y outliers |

**Siguiente notebook:** `02_feature_engineering.ipynb` — transformaciones, agregaciones y documentación de decisiones de ingeniería de features.